# Question 1

In [1]:
import pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

print(spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/15 20:58:17 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


4.1.1


# Question 2

In [2]:
df_yellow_nov_2025 = spark.read.parquet("data/yellow_tripdata_2025-11.parquet")


In [3]:
df_yellow_nov_2025.repartition(4) \
        .write.parquet("data/yellow_tripdata_2025-11_repartition/", mode="overwrite")

In [6]:
!ls -lsah data/yellow_tripdata_2025-11_repartition/

total 207368
    0 -rw-r--r--@  1 nicolenair  staff     0B Mar 15 21:07 _SUCCESS
    0 drwxr-xr-x@ 12 nicolenair  staff   384B Mar 15 21:07 .
    8 -rw-r--r--@  1 nicolenair  staff     8B Mar 15 21:07 ._SUCCESS.crc
    0 drwxr-xr-x@  5 nicolenair  staff   160B Mar 15 21:07 ..
  512 -rw-r--r--@  1 nicolenair  staff   195K Mar 15 21:07 .part-00000-c62aa8d5-4aaf-4457-bf87-70a9a1254343-c000.snappy.parquet.crc
  512 -rw-r--r--@  1 nicolenair  staff   195K Mar 15 21:07 .part-00001-c62aa8d5-4aaf-4457-bf87-70a9a1254343-c000.snappy.parquet.crc
  512 -rw-r--r--@  1 nicolenair  staff   195K Mar 15 21:07 .part-00002-c62aa8d5-4aaf-4457-bf87-70a9a1254343-c000.snappy.parquet.crc
  512 -rw-r--r--@  1 nicolenair  staff   195K Mar 15 21:07 .part-00003-c62aa8d5-4aaf-4457-bf87-70a9a1254343-c000.snappy.parquet.crc
51328 -rw-r--r--@  1 nicolenair  staff    24M Mar 15 21:07 part-00000-c62aa8d5-4aaf-4457-bf87-70a9a1254343-c000.snappy.parquet
51328 -rw-r--r--@  1 nicolenair  staff    24M Mar 15 21:07 part-0000

# Question 3

In [10]:
df_yellow_nov_2025.createOrReplaceTempView("df_yellow_nov_2025")
count_df = spark.sql("SELECT count(*) FROM df_yellow_nov_2025 where DATE_FORMAT(tpep_pickup_datetime, 'yyyy-MM-dd') = '2025-11-15'")
count_df.show()

+--------+
|count(1)|
+--------+
|  162604|
+--------+



# Question 4

In [16]:
longest_trip = spark.sql("SELECT MAX(UNIX_TIMESTAMP(tpep_dropoff_datetime) - UNIX_TIMESTAMP(tpep_pickup_datetime))/3600 FROM df_yellow_nov_2025")
longest_trip.show(truncate=False)

+--------------------------------------------------------------------------------------------------------------------------------------+
|(max((unix_timestamp(tpep_dropoff_datetime, yyyy-MM-dd HH:mm:ss) - unix_timestamp(tpep_pickup_datetime, yyyy-MM-dd HH:mm:ss))) / 3600)|
+--------------------------------------------------------------------------------------------------------------------------------------+
|90.64666666666666                                                                                                                     |
+--------------------------------------------------------------------------------------------------------------------------------------+



# Question 5

4040

# Question 6

In [17]:
# !wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

--2026-03-15 21:24:27--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 18.172.0.48, 18.172.0.77, 18.172.0.12, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|18.172.0.48|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12331 (12K) [text/csv]
Saving to: ‘taxi_zone_lookup.csv’

taxi_zone_lookup.cs 100%[===================>]  12.04K  --.-KB/s    in 0s      

2026-03-15 21:24:28 (1.64 GB/s) - ‘taxi_zone_lookup.csv’ saved [12331/12331]



In [24]:
df_zones = spark.read.csv("data/taxi_zone_lookup.csv", header=True)
df_zones.createOrReplaceTempView("df_zones")

In [25]:
df_zones.head(5)

[Row(LocationID='1', Borough='EWR', Zone='Newark Airport', service_zone='EWR'),
 Row(LocationID='2', Borough='Queens', Zone='Jamaica Bay', service_zone='Boro Zone'),
 Row(LocationID='3', Borough='Bronx', Zone='Allerton/Pelham Gardens', service_zone='Boro Zone'),
 Row(LocationID='4', Borough='Manhattan', Zone='Alphabet City', service_zone='Yellow Zone'),
 Row(LocationID='5', Borough='Staten Island', Zone='Arden Heights', service_zone='Boro Zone')]

In [26]:
df_yellow_nov_2025.head(5)

[Row(VendorID=7, tpep_pickup_datetime=datetime.datetime(2025, 11, 1, 0, 13, 25), tpep_dropoff_datetime=datetime.datetime(2025, 11, 1, 0, 13, 25), passenger_count=1, trip_distance=1.68, RatecodeID=1, store_and_fwd_flag='N', PULocationID=43, DOLocationID=186, payment_type=1, fare_amount=14.9, extra=0.0, mta_tax=0.5, tip_amount=1.5, tolls_amount=0.0, improvement_surcharge=1.0, total_amount=22.15, congestion_surcharge=2.5, Airport_fee=0.0, cbd_congestion_fee=0.75),
 Row(VendorID=2, tpep_pickup_datetime=datetime.datetime(2025, 11, 1, 0, 49, 7), tpep_dropoff_datetime=datetime.datetime(2025, 11, 1, 1, 1, 22), passenger_count=1, trip_distance=2.28, RatecodeID=1, store_and_fwd_flag='N', PULocationID=142, DOLocationID=237, payment_type=1, fare_amount=14.2, extra=1.0, mta_tax=0.5, tip_amount=4.99, tolls_amount=0.0, improvement_surcharge=1.0, total_amount=24.94, congestion_surcharge=2.5, Airport_fee=0.0, cbd_congestion_fee=0.75),
 Row(VendorID=1, tpep_pickup_datetime=datetime.datetime(2025, 11, 1,

In [30]:
least_freq_zone = spark.sql("""
select Zone, COUNT(Zone) from df_yellow_nov_2025 join df_zones ON df_yellow_nov_2025.PULocationID = df_zones.LocationID
GROUP BY Zone
ORDER BY COUNT(Zone) ASC LIMIT 1
""")
least_freq_zone.show(truncate=False)

+---------------------------------------------+-----------+
|Zone                                         |count(Zone)|
+---------------------------------------------+-----------+
|Governor's Island/Ellis Island/Liberty Island|1          |
+---------------------------------------------+-----------+

